In [20]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os, tqdm

sj_path = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/targetSH.json"

method = ["TR", "hou_21", "hou_22", "DiffusionRig", "IC-Light", "Relipa", "ours_256_DiFaReli", "ours_difareli++_oneshot_dstC"]
sample = json.load(open(sj_path, "r"))
meta = json.load(open("./ffhq_targetSH.json", "r"))
os.makedirs("./targetSH_figures/", exist_ok=True)

counter = 0
for pid, dat in tqdm.tqdm(sample['pair'].items()):
    src = dat['src']
    dst = dat['dst']

    out = []
    for m in method:
        meta_dat = meta[m]
        img_dir = meta_dat['img_dir']
        itp_method = meta_dat['itp_method']
        diff_step = meta_dat['diff_step']
        n_frame = int(meta_dat['n_frame'])

        if m in ["TR", "hou_21", "hou_22"]:
            relit_img = f'{img_dir}/src={src}/dst={dst}/res_frame{n_frame-1}.png'
        elif m in ["Relipa"]:
            relit_img = f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/gs=4.5_ds=25/n_frames={n_frame}/256/res_frame{n_frame-1:03d}.png'
        elif m in ["DiffusionRig"]:
            relit_img = f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame}/res_frame{n_frame-1:03d}.png'
        else:
            relit_img = f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame}/res_frame{n_frame-1}.png'
        
        if not os.path.exists(relit_img):
            relit_img = Image.fromarray(np.zeros((256, 256, 3), dtype=np.uint8))
        else:
            relit_img = Image.open(relit_img)
        out.append(relit_img)

        if m == "ours_difareli++_oneshot_dstC":
            shadm = f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame}/dst_shadm_shad_frame{n_frame-1}.png'
            ren = f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame}/dst_ren_frame{n_frame-1}.png'
            shadm = Image.open(shadm)
            ren = Image.open(ren)
            out_cond = np.concatenate([ren, shadm], axis=0)

    out = np.concatenate(out, axis=1)

    Image.fromarray(out).save(f"./targetSH_figures/{pid}_res.png")
    Image.fromarray(out_cond).save(f"./targetSH_figures/{pid}_cond.png")
    # counter += 1
    # if counter >= 5:
    #     break

100%|██████████| 31/31 [00:05<00:00,  6.11it/s]


# Gen final grid

In [ ]:
grid = [
        [2302, 2861, 286, '_unknown', 2913, 1587, 1640],
        [1264, 553, 2554, 1901, 1494, 2015, 1016],
        [2059, 1186, 1355, 1581, 1812, 1838, 2076],
        [1060, 1566, 2245, 2285, 2317, 2412, 2512]
    ]
img_path = '/data/mint/DPM_Dataset/ffhq_256_with_anno/ffhq_256/valid/'
os.makedirs("./targetSH_figures_final_grid/", exist_ok=True)

# for i, each_grid in enumerate(grid):
#     grid = []
#     for e in each_grid:
#             img = Image.open(f"./targetSH_figures/pair{e}_res.png")
#             cond = Image.open(f"./targetSH_figures/pair{e}_cond.png")
#             cond = cond.resize((cond.size[0]//2, cond.size[1]//2))
#             # print(np.array(cond).shape)
#             img = np.concatenate([np.array(img), np.array(cond)], axis=1)
#             grid.append(img)
#     grid = np.concatenate(grid, axis=0)
#     Image.fromarray(grid).save(f"./targetSH_figures_final_grid/grid_row{i}.png")